# Robot ROI Crop Before DINOv2

Use this notebook to test the improved vision flow:

1. Load the uploaded inspection image.
2. Crop only the robot / inspected asset region.
3. Remove most background pipes, walls, windows, and plant structure.
4. Run the existing DINOv2 anomaly localization on the cropped robot image.

This is a notebook workflow first. After the crop behavior looks good, the same logic can be moved into `app/backend/vision_runtime.py` before `score_image()` runs.

In [ ]:
# Run this cell once if these packages are missing in your notebook environment.
%pip install opencv-python pillow matplotlib numpy torch transformers

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

APP_ROOT = Path.cwd() if Path.cwd().name == 'app' else Path.cwd() / 'app'
BACKEND_ROOT = APP_ROOT / 'backend'
sys.path.insert(0, str(BACKEND_ROOT))

from vision_runtime import infer_fault_from_path, score_image

WORK_DIR = APP_ROOT / 'runtime' / 'roi_notebook'
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('APP_ROOT:', APP_ROOT)
print('WORK_DIR:', WORK_DIR)

## 1. Select Image

Change `IMAGE_PATH` to the uploaded robot image you want to test. Use a file under `app/runtime/runs/.../inputs/...` or copy an image into `app/test_inputs/`.

In [ ]:
# Portable test images live in user_test/ so this notebook works after clone.
# Change TEST_IMAGE_NAME to another committed test image when needed.
REPO_ROOT = APP_ROOT.parent
TEST_IMAGE_NAME = 'robot_corrosion.jpg'
IMAGE_PATH = REPO_ROOT / 'user_test' / TEST_IMAGE_NAME

if not IMAGE_PATH.is_file():
    user_test_dir = REPO_ROOT / 'user_test'
    candidates = sorted(user_test_dir.glob('*')) if user_test_dir.exists() else []
    image_candidates = [p for p in candidates if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff'}]
    if image_candidates:
        IMAGE_PATH = image_candidates[0]

if not IMAGE_PATH.is_file():
    candidates = sorted((APP_ROOT / 'runtime' / 'runs').glob('*/inputs/*')) if (APP_ROOT / 'runtime' / 'runs').exists() else []
    image_candidates = [p for p in candidates if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff'}]
    if image_candidates:
        IMAGE_PATH = image_candidates[-1]

print('IMAGE_PATH:', IMAGE_PATH)
assert IMAGE_PATH.is_file(), 'Set IMAGE_PATH to a readable image file.'

image = cv2.imread(str(IMAGE_PATH))
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 5))
plt.imshow(image_rgb)
plt.axis('off')
plt.title('Original uploaded image')
plt.show()

## 2. Option A: Manual Robot Box

This is the most reliable method for a demo. Set the box around the robot only.

Format: `x1, y1, x2, y2` in image pixels.

In [ ]:
# Change these numbers for your image.
# Use the plotted image above to estimate the robot bounding box.
MANUAL_BOX = None
# Example:
# MANUAL_BOX = (40, 20, 260, 190)

def crop_with_box(rgb: np.ndarray, box):
    h, w = rgb.shape[:2]
    x1, y1, x2, y2 = box
    x1 = max(0, min(w - 1, int(x1)))
    y1 = max(0, min(h - 1, int(y1)))
    x2 = max(x1 + 1, min(w, int(x2)))
    y2 = max(y1 + 1, min(h, int(y2)))
    return rgb[y1:y2, x1:x2], (x1, y1, x2, y2)

manual_crop = None
if MANUAL_BOX is not None:
    manual_crop, manual_box = crop_with_box(image_rgb, MANUAL_BOX)
    plt.figure(figsize=(6, 5))
    plt.imshow(manual_crop)
    plt.axis('off')
    plt.title(f'Manual robot crop: {manual_box}')
    plt.show()
else:
    print('MANUAL_BOX is None. The notebook will use automatic foreground crop below.')

## 3. Option B: Automatic Foreground Crop

This uses OpenCV GrabCut to remove background and find the biggest foreground object. It is not perfect, but it is useful when you do not want to manually enter a box.

In [ ]:
def automatic_robot_crop(rgb: np.ndarray, margin_ratio: float = 0.08):
    h, w = rgb.shape[:2]
    rect = (
        max(1, int(w * 0.08)),
        max(1, int(h * 0.02)),
        max(2, int(w * 0.84)),
        max(2, int(h * 0.96)),
    )
    mask = np.zeros((h, w), np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    cv2.grabCut(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
    foreground = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
    kernel = np.ones((9, 9), np.uint8)
    foreground = cv2.morphologyEx(foreground, cv2.MORPH_CLOSE, kernel)
    foreground = cv2.morphologyEx(foreground, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(foreground, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        x, y, rw, rh = rect
        return rgb[y:y + rh, x:x + rw], (x, y, x + rw, y + rh), foreground

    large = max(contours, key=cv2.contourArea)
    x, y, cw, ch = cv2.boundingRect(large)
    margin = int(max(cw, ch) * margin_ratio)
    x1 = max(0, x - margin)
    y1 = max(0, y - margin)
    x2 = min(w, x + cw + margin)
    y2 = min(h, y + ch + margin)
    return rgb[y1:y2, x1:x2], (x1, y1, x2, y2), foreground

auto_crop, auto_box, auto_mask = automatic_robot_crop(image_rgb)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(auto_mask, cmap='gray')
axes[0].set_title('Foreground mask')
axes[0].axis('off')
axes[1].imshow(auto_crop)
axes[1].set_title(f'Automatic crop: {auto_box}')
axes[1].axis('off')
plt.show()

## 4. Choose Crop And Run DINOv2

If `MANUAL_BOX` is set, the notebook uses the manual crop. Otherwise it uses the automatic crop. The DINOv2 result is generated only on the cropped robot image.

In [ ]:
chosen_crop = manual_crop if manual_crop is not None else auto_crop
chosen_box = manual_box if manual_crop is not None else auto_box

fault = infer_fault_from_path(str(IMAGE_PATH), fallback='visual_anomaly')
crop_path = WORK_DIR / f'{IMAGE_PATH.stem}_robot_crop.png'
result_path = WORK_DIR / f'{IMAGE_PATH.stem}_robot_crop_dinov2.png'

Image.fromarray(chosen_crop).save(crop_path)
metrics = score_image(crop_path, result_path, fault)

print('fault:', fault)
print('crop_box:', chosen_box)
print('crop_path:', crop_path)
print('result_path:', result_path)
print('anomaly_score:', metrics['anomaly_score'])
print('is_anomaly:', metrics['is_anomaly'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(chosen_crop)
axes[0].set_title('Robot-only crop sent to DINOv2')
axes[0].axis('off')
axes[1].imshow(Image.open(result_path))
axes[1].set_title('DINOv2 defect localization on crop')
axes[1].axis('off')
plt.show()

## 5. How To Move This Into Backend

After the crop looks correct, production flow should become:

```text
uploaded image -> robot ROI crop -> DINOv2 score_image(crop) -> mapped/displayed defect overlay
```

Implementation place:

```text
app/backend/vision_runtime.py
```

Add a preprocessing step before `score_image()` in `build_vision_result()`:

```python
robot_crop_path = crop_robot_region(image_file, run_dir)
metrics = score_image(robot_crop_path, heatmap, predicted_fault)
```

For best accuracy in real plant photos, prefer a manual or UI-assisted ROI box because automatic foreground segmentation can still fail when the background is complex.